# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


def load_digits_data():
    digits = load_digits()
    X = digits.images.astype(np.float32) / 16.0
    y = digits.target.astype(np.int32)
    X = X[..., None]

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.165, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )

    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True) + 1e-8

    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = load_digits_data()
input_shape = X_train.shape[1:]
num_classes = 10
device = '/CPU:0'
print_every = 20

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


Train data shape:  (1200, 8, 8, 1)
Train labels shape:  (1200,) int32
Validation data shape:  (300, 8, 8, 1)
Validation labels shape:  (300,)
Test data shape:  (297, 8, 8, 1)
Test labels shape:  (297,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 8, 8, 1) (64,)
1 (64, 8, 8, 1) (64,)
2 (64, 8, 8, 1) (64,)
3 (64, 8, 8, 1) (64,)
4 (64, 8, 8, 1) (64,)
5 (64, 8, 8, 1) (64,)
6 (64, 8, 8, 1) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(channel_1, 5, padding='same', activation='relu', kernel_initializer=initializer)
        self.conv2 = tf.keras.layers.Conv2D(channel_2, 3, padding='same', activation='relu', kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax', kernel_initializer=initializer)

    def call(self, x, training=False):
        if len(x.shape) == 4 and x.shape[1] in (1, 3) and x.shape[-1] not in (1, 3):
            x = tf.transpose(x, [0, 2, 3, 1])
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)
        return scores


In [7]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()


(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                gradients = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                train_loss.update_state(loss)
                train_accuracy.update_state(y_np, scores)
                t += 1

            val_loss.reset_state()
            val_accuracy.reset_state()
            for test_x, test_y in val_dset:
                prediction = model(test_x, training=False)
                t_loss = loss_fn(test_y, prediction)
                val_loss.update_state(t_loss)
                val_accuracy.update_state(test_y, prediction)

            template = 'Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
            print(template.format(
                epoch + 1, train_loss.result(), train_accuracy.result() * 100,
                val_loss.result(), val_accuracy.result() * 100
            ))

        return model


In [9]:
hidden_size, num_classes = 256, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

model_fc_subclass = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.591310501098633, Accuracy: 9.375, Val Loss: 2.4012413024902344, Val Accuracy: 17.66666603088379


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    return model

def optimizer_init_fn():
    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    return optimizer

model_cnn_subclass = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.9381215572357178, Accuracy: 25.0, Val Loss: 2.803943395614624, Val Accuracy: 11.666666984558105


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    hidden_layer_size, num_classes = 256, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

model_fc_seq = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.167064666748047, Accuracy: 17.1875, Val Loss: 3.1681485176086426, Val Accuracy: 9.333333015441895


Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

 1/19 ━━━━━━━━━━━━━━━━━━━━ 8s 481ms/step - loss: 3.3432 - sparse_categorical_accuracy: 0.1094

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.3223 - sparse_categorical_accuracy: 0.2392 - val_loss: 1.8231 - val_sparse_categorical_accuracy: 0.4000


 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 2.0005 - sparse_categorical_accuracy: 0.3438

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9017 - sparse_categorical_accuracy: 0.3468 


[1.90170156955719, 0.3468013405799866]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(32, 5, padding='same', activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer),
    ])
    return model

learning_rate = 3e-3
def optimizer_init_fn():
    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    return optimizer

model_cnn_seq = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.0900046825408936, Accuracy: 7.8125, Val Loss: 2.902639865875244, Val Accuracy: 6.6666669845581055


In [14]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)


 1/19 ━━━━━━━━━━━━━━━━━━━━ 12s 706ms/step - loss: 2.9808 - sparse_categorical_accuracy: 0.0938

12/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.6910 - sparse_categorical_accuracy: 0.1016   

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 2.3385 - sparse_categorical_accuracy: 0.1875 - val_loss: 1.9479 - val_sparse_categorical_accuracy: 0.3667


 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 2.0390 - sparse_categorical_accuracy: 0.3750

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.9596 - sparse_categorical_accuracy: 0.3670 


[1.9595637321472168, 0.3670033812522888]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [16]:
hidden_size, num_classes = 256, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

model_fc_func = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.58007550239563, Accuracy: 10.9375, Val Loss: 2.6146187782287598, Val Accuracy: 17.0


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same', kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPool2D(pool_size=2)
        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPool2D(pool_size=2, padding='same')
        self.flatten = tf.keras.layers.Flatten()
        self.dropout = tf.keras.layers.Dropout(0.3)
        self.fc1 = tf.keras.layers.Dense(128, activation='relu', kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

    def call(self, input_tensor, training=False):
        x = input_tensor
        if len(x.shape) == 4 and x.shape[1] in (1, 3) and x.shape[-1] not in (1, 3):
            x = tf.transpose(x, [0, 2, 3, 1])
        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.dropout(x, training=training)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


print_every = 20
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

model_custom = train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)


Iteration 0, Epoch 1, Loss: 3.1134300231933594, Accuracy: 12.5, Val Loss: 3.54272723197937, Val Accuracy: 11.666666984558105


Iteration 20, Epoch 2, Loss: 1.01780366897583, Accuracy: 67.1875, Val Loss: 1.1889164447784424, Val Accuracy: 55.66666793823242


Iteration 40, Epoch 3, Loss: 0.510817289352417, Accuracy: 86.97917175292969, Val Loss: 0.5363549590110779, Val Accuracy: 84.0


Iteration 60, Epoch 4, Loss: 0.3782910108566284, Accuracy: 87.5, Val Loss: 0.27228161692619324, Val Accuracy: 92.33333587646484


Iteration 80, Epoch 5, Loss: 0.1641659438610077, Accuracy: 95.9375, Val Loss: 0.1813240796327591, Val Accuracy: 95.66666412353516


Iteration 100, Epoch 6, Loss: 0.16128341853618622, Accuracy: 94.27082824707031, Val Loss: 0.15501129627227783, Val Accuracy: 96.33332824707031


Iteration 120, Epoch 7, Loss: 0.12489645183086395, Accuracy: 96.875, Val Loss: 0.11615218967199326, Val Accuracy: 97.33333587646484


Iteration 140, Epoch 8, Loss: 0.09999098628759384, Accuracy: 97.265625, Val Loss: 0.09065460413694382, Val Accuracy: 97.66667175292969


Iteration 160, Epoch 9, Loss: 0.08102762699127197, Accuracy: 96.875, Val Loss: 0.08488748222589493, Val Accuracy: 97.33333587646484


Iteration 180, Epoch 10, Loss: 0.0625653937458992, Accuracy: 98.75, Val Loss: 0.09638570249080658, Val Accuracy: 96.66666412353516


Опишите все эксперименты, результаты. Сделайте выводы.

In [ ]:
test_loss, test_acc = model_custom.evaluate(X_test, y_test, verbose=0)
print('???????? accuracy ?????? ??????: %.4f' % test_acc)
print()
print('?????: ? ???? ?????? ???? ????????? ????????? ????????? ??????? ? TensorFlow. ??????? ???????????? ? ??????????? ?????????? ???? ????? ????? ????? ???????? ???????? ????????? ????????, ? ?????? ????????? ???? ????? ???????? ?????????? ???? ? Batch Normalization, pooling ? Dropout. ?? 10 ???? ???????? ??? ???????? accuracy 0.9667 ?? ????????????? ??????? ? 0.9731 ?? ???????? ???????, ??????? ??? ?????? ?????? ?????????? ??????????? ????????? ???????? ??????????.')
